# Exercise: Feed Clean Documents into Your RAG Pipeline

In the walkthrough you built the RAG pipeline by hand on text pulled straight from a PDF reader. Real documents are messier than that, and the quality of the text you feed in directly shapes the quality of what you retrieve.

In this exercise you will reuse the previous pipeline but we convert the document to Markdown with
[MarkItDown](https://github.com/microsoft/markitdown), clean it, and then **measure** how much that cleaning changes what the pipeline retrieves and answers.

## Learning Objectives

By completing this exercise, you will be able to:

- Convert a PDF to Markdown with MarkItDown and inspect the raw conversion.
- Write cleaning rules that remove common extraction artifacts.
- Measure the effect of cleaning on retrieval with similarity scores and chunk overlap.

## Setup

Load your credentials. The Groq client reads `GROQ_API_KEY` from your `.env` file.

In [1]:
from dotenv import load_dotenv

load_dotenv()

PDF = "documents/react_paper.pdf"
QUERY = "What is the ReAct framework and how does it combine reasoning and acting?"

## Provided: the RAG pipeline from the walkthrough

Run this cell as is. It gives you two things:

- `run_rag(text, query)`, which chunks a body of text, embeds it, retrieves the most
  relevant chunks, and generates an answer. It returns the `answer`, the retrieved
  `chunks`, and their cosine similarity `scores`.
- `retrieval_overlap(chunks_a, chunks_b)`, the Jaccard overlap between two sets of
  retrieved chunks (0 means completely different, 1 means identical).

You do not need to change anything here.

In [2]:
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


def chunk_text(text, size=800, overlap=100):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]


def retrieve(query, chunks, chunk_vectors, k=3):
    query_vector = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vectors @ query_vector
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]


def generate_answer(query, context):
    prompt = (
        "Answer the question using only the context below.\n\n"
        "Context:\n" + "\n---\n".join(context) + f"\n\nQuestion: {query}\nAnswer:"
    )
    response = Groq().chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


def run_rag(text, query, k=3):
    """Chunk, embed, retrieve, and generate. Returns answer, chunks, and scores."""
    chunks = chunk_text(text)
    chunk_vectors = embedder.encode(chunks, normalize_embeddings=True)
    hits = retrieve(query, chunks, chunk_vectors, k=k)
    context = [chunk for chunk, _ in hits]
    return {
        "answer": generate_answer(query, context),
        "chunks": context,
        "scores": [score for _, score in hits],
    }


def retrieval_overlap(chunks_a, chunks_b):
    """Jaccard overlap between two lists of retrieved chunks (0 to 1)."""
    a, b = set(chunks_a), set(chunks_b)
    return len(a & b) / len(a | b) if (a or b) else 0.0

/home/hashim/all_imp_repos/ds-rag-pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Exercise 1: Convert the document to Markdown

Use MarkItDown to convert the PDF to a Markdown string.

Hints:

- The API is `MarkItDown().convert(path).text_content`.
- After converting, print the first 1000 characters and look closely. Real PDF
  extractions often have missing spaces, repeated page headers, and stray line breaks.
  Note the problems you see: you will fix them next.

In [3]:
from markitdown import MarkItDown


def to_markdown(path):
    """Convert a document to a Markdown string with MarkItDown."""
    return MarkItDown().convert(path).text_content

In [4]:
raw_text = to_markdown(PDF)
print(raw_text[:1000])

PublishedasaconferencepaperatICLR2023
REACT: SYNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
ShunyuYao∗*,1,JeffreyZhao2,DianYu2,NanDu2,IzhakShafran2,KarthikNarasimhan1,YuanCao2
1DepartmentofComputerScience,PrincetonUniversity
2GoogleResearch,Brainteam
1{shunyuy,karthikn}@princeton.edu
2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com
ABSTRACT
Whilelargelanguagemodels(LLMs)havedemonstratedimpressiveperformance
across tasks in language understanding and interactive decision making, their
abilitiesforreasoning(e.g. chain-of-thoughtprompting)andacting(e.g. action
plangeneration)haveprimarilybeenstudiedasseparatetopics. Inthispaper,we
exploretheuseofLLMstogeneratebothreasoningtracesandtask-specificactions
inaninterleavedmanner,allowingforgreatersynergybetweenthetwo: reasoning
traces help the model induce, track, and update action plans as well as handle
exceptions,whileactionsallowittointerfacewithandgatheradditionalinformation
fromexternalsourcessuchasknowledgebasesorenvironments. W

## Exercise 2: Clean the Markdown

Write a cleaning function that fixes the problems you spotted. Decide which rules
matter for this document.

Ideas to consider:

- Collapse runs of blank lines and strip trailing whitespace.
- Drop lines that are just a page number or the repeated conference header.
- Remove very short lines that are page furniture rather than content.
- Rejoin words split across a line break by a hyphen.

Store the result as `clean_text` and print the character count before and after so you
can see how much you removed.

In [5]:
import re

HEADER = "PublishedasaconferencepaperatICLR2023"


def clean_markdown(text):
    """Return a cleaned version of the Markdown text."""
    # Rejoin words split across a line break by a hyphen ("synergiz-\ning" -> "synergizing")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    lines = []
    for line in text.splitlines():
        line = line.rstrip()
        stripped = line.strip()
        # Drop the running conference header repeated on every page
        if stripped == HEADER:
            continue
        # Drop lines that are just a page number
        if re.fullmatch(r"\d{1,3}", stripped):
            continue
        # Drop Markdown table separator rows such as "| --- | --- |"
        if stripped and re.fullmatch(r"[|\-\s:]+", stripped):
            continue
        # Drop very short lines that are page furniture rather than content
        if 0 < len(stripped) <= 2:
            continue
        lines.append(line)
    text = "\n".join(lines)

    # Restore the space missing after punctuation (",we" -> ", we"), skipping decimals like 3.5
    text = re.sub(r"([,;:!?])(?=[A-Za-z])", r"\1 ", text)
    text = re.sub(r"(?<=[a-z])\.(?=[A-Z][a-z])", ". ", text)

    # Collapse runs of spaces and of blank lines
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [6]:
clean_text = clean_markdown(raw_text)

removed = len(raw_text) - len(clean_text)
print(f"raw:   {len(raw_text):,} characters")
print(f"clean: {len(clean_text):,} characters")
print(f"removed {removed:,} characters ({removed / len(raw_text):.1%})\n")
print(clean_text[:1000])

raw:   205,117 characters
clean: 157,835 characters
removed 47,282 characters (23.1%)

REACT: SYNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
ShunyuYao∗*,1, JeffreyZhao2, DianYu2, NanDu2, IzhakShafran2, KarthikNarasimhan1, YuanCao2
1DepartmentofComputerScience, PrincetonUniversity
2GoogleResearch, Brainteam
1{shunyuy, karthikn}@princeton.edu
2{jeffreyzhao, dianyu, dunan, izhak, yuancao}@google.com
ABSTRACT
Whilelargelanguagemodels(LLMs)havedemonstratedimpressiveperformance
across tasks in language understanding and interactive decision making, their
abilitiesforreasoning(e.g. chain-of-thoughtprompting)andacting(e.g. action
plangeneration)haveprimarilybeenstudiedasseparatetopics. Inthispaper, we
exploretheuseofLLMstogeneratebothreasoningtracesandtask-specificactions
inaninterleavedmanner, allowingforgreatersynergybetweenthetwo: reasoning
traces help the model induce, track, and update action plans as well as handle
exceptions, whileactionsallowittointerfacewithandgatheradditionalinf

## Exercise 3: Measure the effect of cleaning

This is the payoff. Run the provided pipeline twice on the same question, once with
`raw_text` and once with `clean_text`, then compare them with two metrics.

**Metric 1: mean retrieval score.** Average the `scores` of the retrieved chunks for
each run. A higher mean suggests the retrieved chunks match the query more closely.

**Metric 2: retrieval overlap.** Use `retrieval_overlap(...)` on the two `chunks`
lists. A low overlap means cleaning changed *which* chunks were retrieved.

> Read the scores as a signal, not a verdict. The raw and cleaned runs have different
> chunk sets, so the numbers are not a perfect like-for-like comparison. Always read the
> retrieved chunks and the answer alongside the metrics.

Tasks:

- Call `run_rag` on `raw_text` and on `clean_text`.
- Print the mean retrieval score for each run and the overlap between them.
- Print both answers, then write a sentence on what changed and why.

In [7]:
raw_run = run_rag(raw_text, QUERY)
clean_run = run_rag(clean_text, QUERY)

print(f"mean retrieval score (raw):   {np.mean(raw_run['scores']):.4f}")
print(f"mean retrieval score (clean): {np.mean(clean_run['scores']):.4f}")
print(f"retrieval overlap: {retrieval_overlap(raw_run['chunks'], clean_run['chunks']):.2f}")

print("\n=== Raw answer ===\n" + raw_run["answer"])
print("\n=== Clean answer ===\n" + clean_run["answer"])

mean retrieval score (raw):   0.5104
mean retrieval score (clean): 0.5533
retrieval overlap: 0.00

=== Raw answer ===
**ReAct** is a framework that lets a large language model *reason* and *act* in a single, interleaved loop.  
- The model writes a **thought** in natural language, then immediately follows it with an **action** (e.g., a query, a move in a game, a web‑navigation step).  
- This “thought‑action” pattern is the core of the prompt design: annotators simply type their thoughts on top of the actions they would take, without any special formatting or example selection.  
- Because the thought space is flexible and the action space can be any task‑specific set, ReAct works for a wide range of problems such as question answering, fact verification, text‑based games, and web navigation.  
- The framework relies on the model’s integrated decision‑making and reasoning abilities, and it can incorporate external knowledge bases to ground its reasoning.  
- In practice, ReAct outperfo

**What changed:** cleaning raised the mean retrieval score from about 0.51 to 0.55, and the
retrieval overlap was 0: the two runs retrieved completely different chunks. That is expected,
because removing the repeated header, page numbers and table separators shifts every 800-character
chunk boundary, so the chunk sets are no longer comparable one-to-one. Both answers describe the
same thought, action and observation loop. The cleaned answer is slightly tighter and drops the
less relevant "IM-style prompting" comparison that came from a noisier chunk.

## Exercise 4 (stretch): Score the answers with an LLM judge

Similarity scores tell you about retrieval, not about the final answer. A common way to
score answer quality without a reference answer is to use the model as a judge.

Write a function that asks the model to rate, from 1 to 5, how well an answer is
supported by the context it was given (its faithfulness). Run it for both the raw and
the cleaned answer and compare.

Hint: build a prompt that includes the question, the context, and the answer, and ask
the model to reply with a single number. Reuse the `Groq` client from the provided cell.

In [8]:
def judge_faithfulness(question, context, answer):
    """Ask the model to rate 1 to 5 how well the answer is supported by the context."""
    prompt = (
        "You are grading a question-answering system.\n"
        "Rate from 1 to 5 how well the ANSWER is supported by the CONTEXT "
        "(5 = every claim is supported by the context, 1 = mostly unsupported).\n"
        "Reply with a single digit only.\n\n"
        "CONTEXT:\n" + "\n---\n".join(context) + "\n\n"
        f"QUESTION: {question}\n\nANSWER: {answer}\n\nSCORE:"
    )
    response = Groq().chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    match = re.search(r"[1-5]", response.choices[0].message.content)
    return int(match.group()) if match else None

In [9]:
raw_score = judge_faithfulness(QUERY, raw_run["chunks"], raw_run["answer"])
clean_score = judge_faithfulness(QUERY, clean_run["chunks"], clean_run["answer"])

print(f"faithfulness (raw):   {raw_score}/5")
print(f"faithfulness (clean): {clean_score}/5")

faithfulness (raw):   5/5
faithfulness (clean): 5/5


## Conclusion and Reflection

- Which cleaning rule made the biggest difference to the mean retrieval score or the
  overlap, and why?
- Did cleaning change the answer, or only the chunks behind it? What does that tell you
  about how sensitive this question is to noise?
- The mean score is an imperfect metric. What would you need in order to measure answer
  quality properly, and which tools (for example RAGAS) formalise this?

**My answers**

- **Biggest effect:** removing the running header `PublishedasaconferencepaperatICLR2023`
  (33 copies) together with the bare page numbers and the Markdown table separator rows. These
  account for most of the 23% of characters removed. They carry no meaning, but they dilute the
  embedding of every chunk they land in, so removing them is what lifts the similarity scores.
- **Answer vs. chunks:** cleaning changed the chunks completely (overlap 0) but barely changed the
  answer, and the LLM judge gave both 5/5 for faithfulness. This question is broad and the paper
  explains ReAct in many places, so almost any relevant chunk is enough. A narrow factual
  question, such as a specific benchmark number, would be far more sensitive to noise.
- **Measuring properly:** we would need a labelled test set (questions with reference answers and
  the passages that support them) to measure retrieval precision and recall and answer correctness.
  Tools such as RAGAS formalise this with metrics like faithfulness, answer relevancy, context
  precision and context recall. Note that the remaining glued-together words (`Inthispaper`) are
  a limit of this PDF's text layer. Fixing them would need a word segmenter or a different
  extractor, not regex rules.